In [2]:
from pathlib import Path
import sys

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))



In [3]:
import torch
from torch import nn

class DiscriminatorCNNVersion1(nn.Module):
    # CNN network that combines temporal data (log spectrogram)
    # and summarized audio data such as MFCCs, zero crossings and so forth at the linear layer.

    def __init__(self, handcrafted_features=68):

        super().__init__()  # inherit nn module

        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=16,
            kernel_size=3,
            padding=1)  # first layer convolution

        self.relu = nn.ReLU()

        self.pool = nn.MaxPool2d(kernel_size=2)  # reduce dimensionality

        # to further reduce the size of the huge vector
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))

        self.flatten = nn.Flatten(start_dim=1)

        # input is 16, 4x4, feature maps for each input
        self.hidden = nn.Linear(16*4*4 + handcrafted_features, 64)

        # output of hidden is.. a 64 dim vector, we output a logit!
        self.out = nn.Linear(64, 1)

    def forward(self, logSpec, compressedFeatures):
        # input -> conv -> activations -> pool -> reduce adaptive pool -> flatten  -> hidden layer -> output
        x = self.conv1(logSpec)
        x = self.relu(x)
        x = self.pool(x)
        if x.device.type == "mps":
            x = self.adaptive_pool(x.cpu()).to(x.device)
        else:
            x = self.adaptive_pool(x)
        x = self.flatten(x)  # (Batch size, 256)

        # combine x and compressed non-temporals
        x = self.hidden(torch.cat([x, compressedFeatures], dim=1))
        x = self.relu(x)  # hidden layer stuff goes through relu again

        return self.out(x)

In [ ]:
class SpectrogramOnlyCNN(DiscriminatorCNNVersion1):
    def __init__(self):
        super().__init__(handcrafted_features=0)

    def forward(self, log_spec):
        x = self.pool(self.relu(self.conv1(log_spec)))
        x = self.adaptive_pool(x)
        x = self.flatten(x)
        x = self.relu(self.hidden(x))
        return self.out(x) # alt version

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
from utils import features as f

torch.manual_seed(42)

EPOCHS = 3
BATCH_SIZE = 8
THRESHOLD = 0.5

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)

data_dir = root / "dataset_NEW"


def load_split(filename):
    # I want to:
    # 1. Load the CSV file into a pandas DataFrame.
    # 2. Extract the features and labels from the DataFrame.
    # 3. Convert the audio files to spectrograms and extract the handcrafted features.
    # 4. Return the spectrograms, handcrafted features, and labels as PyTorch tensors.
    df = pd.read_csv(data_dir / filename)  # load it in

    spectrograms = []
    handcrafted_features = []
    labels = []

    for index, row in df.iterrows():

        y, sr = f.load_audio(data_dir / row['audio_path'])

        log_spectrogram = f.extract_log_mel_spectrogram(y, sr)

        mfcc = f.extract_mfcc(y, sr)
        chroma = f.extract_chroma(y, sr)
        spec_centr = f.extract_spectral_centroid(y, sr)
        zcr = f.extract_zero_crossing_rate(y, sr)

        mfcc_compress = f.compress_features(mfcc)
        chroma_compress = f.compress_features(chroma)
        spec_centr_compress = f.compress_features(spec_centr)
        zcr_compress = f.compress_features(zcr)

        combined = np.concat(
            (mfcc_compress, chroma_compress, spec_centr_compress, zcr_compress))

        labels.append(row["label"])

        spectrograms.append(log_spectrogram)
        handcrafted_features.append(combined)

    spectrograms = torch.from_numpy(
        np.stack(spectrograms)).unsqueeze(1).float()
    handcrafted_features = torch.from_numpy(
        np.stack(handcrafted_features)).float()
    labels = torch.tensor(labels, dtype=torch.float32)

    return spectrograms, handcrafted_features, labels

Device: mps


In [ ]:
from copy import deepcopy
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# features
train_spec, train_hand, train_y = load_split("train.csv")
valid_spec, valid_hand, valid_y = load_split("valid.csv")
test_spec, test_hand, test_y = load_split("test.csv")

# normalize
spec_mean = train_spec.mean()
spec_std = train_spec.std(unbiased=False).clamp_min(1e-6)
hand_mean = train_hand.mean(dim=0)
hand_std = train_hand.std(dim=0, unbiased=False).clamp_min(1e-6)


def make_loader(spec, hand, labels, shuffle=False):
    spec = (spec - spec_mean) / spec_std
    hand = (hand - hand_mean) / hand_std
    return DataLoader(
        TensorDataset(spec, hand, labels),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
    )


train_loader = make_loader(train_spec, train_hand, train_y, shuffle=True)
valid_loader = make_loader(valid_spec, valid_hand, valid_y)
test_loader = make_loader(test_spec, test_hand, test_y)

device = torch.device("cpu")
model = DiscriminatorCNNVersion1().to(device)
loss_function = nn.BCEWithLogitsLoss() # tillåter att skicka in råa logits
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [ ]:

def train(data, model, optim, device, loss_function):
    run_loss, total = 0, 0
    correct = 0

    model.train()
    for spectrogram, handcrafted, labels in tqdm(data, desc=f"Train"):
        spectrogram, handcrafted, labels = spectrogram.to(
            device), handcrafted.to(device), labels.to(device)
        optim.zero_grad()
        output = model(spectrogram, handcrafted).squeeze(-1)
        loss = loss_function(output, labels)
        loss.backward()
        optim.step()

        predicted = torch.sigmoid(output) >= 0.5
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        run_loss += loss.item()*labels.size(0)

    return run_loss/total, correct/total


def evaluate(data, model, loss_function, device):
    model.eval()
    run_loss, total = 0, 0
    correct = 0
    with torch.no_grad():
        for spectrogram, handcrafted, labels in tqdm(data, desc=f"Testing"):
            spectrogram, handcrafted, labels = spectrogram.to(
                device), handcrafted.to(device), labels.to(device)
            output = model(spectrogram, handcrafted).squeeze(-1)
            loss = loss_function(output, labels)
            predicted = torch.sigmoid(output) >= 0.5
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            run_loss += loss.item()*labels.size(0)

    return run_loss/total, correct/total

In [8]:
best_loss = float("inf")

for epoch in range(EPOCHS):
    train_loss, train_acc = train(
        train_loader, model, optimizer, device, loss_function
    )
    val_loss, val_acc = evaluate(
        valid_loader, model, loss_function, device
    )

    if val_loss < best_loss:
        best_loss = val_loss
        best_weights = deepcopy(model.state_dict())

    print(
        f"Epoch {epoch + 1}: "
        f"train loss={train_loss:.4f}, accuracy={train_acc:.1%} | "
        f"val loss={val_loss:.4f}, accuracy={val_acc:.1%}"
    )

Testing: 100%|██████████| 12/12 [00:00<00:00, 15.68it/s]


Epoch 1: train loss=0.2556, accuracy=94.5% | val loss=0.0947, accuracy=96.7%


Testing: 100%|██████████| 12/12 [00:00<00:00, 16.28it/s]


Epoch 2: train loss=0.0489, accuracy=99.5% | val loss=0.0619, accuracy=98.9%


Testing: 100%|██████████| 12/12 [00:00<00:00, 16.03it/s]

Epoch 3: train loss=0.0295, accuracy=99.5% | val loss=0.0523, accuracy=97.8%


In [9]:
model.load_state_dict(best_weights)

test_loss, test_acc = evaluate(
    test_loader, model, loss_function, device
)
print(f"Test loss={test_loss:.4f}, accuracy={test_acc:.1%}")

Testing: 100%|██████████| 18/18 [00:01<00:00, 16.30it/s]

Test loss=0.0791, accuracy=95.7%
